# Figure 5 panel-generation utilities

This notebook reads `data/testing_2_hard_all.xlsx` and generates the high-stiffness validation panels associated with Fig. 5(i–o). Launch Jupyter from the repository root so that relative paths resolve correctly.


In [ ]:
import matplotlib as mpl
mpl.rcParams['xtick.major.size'] = 8
mpl.rcParams['ytick.major.size'] = 8
mpl.rcParams['xtick.major.width'] = 3
mpl.rcParams['ytick.major.width'] = 3

In [ ]:
# Use serif + bold global settings before drawing
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif']  = ['DejaVu Serif', 'Liberation Serif']
plt.rcParams['font.weight'] = 'bold'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# ===== Read data =====
data_path = "data/testing_2_hard_all.xlsx"
df_raw = pd.read_excel(data_path)

# Use all samples (no outlier / non-positive-target filtering)
df = df_raw.copy().reset_index(drop=True)
print(f"Loaded {data_path}: {len(df)} rows (all kept)")
print("Mean Rel. Error (%):", df[["EpH_relative_error", "EppH_relative_error"]].mean().to_dict())
print("Mean Abs. Error (MPa):", df[["EpH_error", "EppH_error"]].mean().to_dict())

# ===== Global font style =====
plt.rcParams.update({
    "font.size": 20,
    "font.weight": "bold",
    "font.family": "serif"
})

def beautify_ax(ax):
    """Beautify axes: thicken spines and ticks"""
    for spine in ax.spines.values():
        spine.set_linewidth(2.5)
    ax.tick_params(axis="both", which="major", width=3, length=8, labelsize=18)

def set_parity_limits(ax, x, y, pad_frac=0.05):
    """Equal-aspect limits that fit both series (hard-material ranges)."""
    lo = float(np.nanmin([np.nanmin(x), np.nanmin(y)]))
    hi = float(np.nanmax([np.nanmax(x), np.nanmax(y)]))
    span = hi - lo
    if span <= 0:
        span = max(abs(hi), 1.0)
    pad = pad_frac * span
    lo_p, hi_p = lo - pad, hi + pad
    if lo_p > 0:
        lo_p = 0.0  # start at 0 for moduli
    ax.set_xlim(lo_p, hi_p)
    ax.set_ylim(lo_p, hi_p)
    ax.set_aspect("equal", adjustable="box")
    ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))

# ==============
# 1) EpH vs EpH_calc
# ==============
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(df["EpH_calc"], df["EpH"], alpha=0.8, s=70, label="Samples")
lo, hi = float(df["EpH_calc"].min()), float(df["EpH_calc"].max())
ax.plot([lo, hi], [lo, hi], "r--", lw=2.5)
ax.set_xlabel("$E_H^{\prime}$ (Calc) (MPa)", fontsize=24, fontweight="bold", fontfamily="serif", labelpad=10)
ax.set_ylabel("$E_H^{\prime}$ (Pred) (MPa)", fontsize=24, fontweight="bold", fontfamily="serif", labelpad=5)
ax.legend(prop={"size": 20, "weight": "bold", "family": "serif"})
set_parity_limits(ax, df["EpH_calc"].values, df["EpH"].values)
beautify_ax(ax)
plt.tight_layout()
plt.show()

# ==============
# 2) EppH vs EppH_calc
# ==============
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(df["EppH_calc"], df["EppH"], alpha=0.8, s=70, label="Samples")
lo, hi = float(df["EppH_calc"].min()), float(df["EppH_calc"].max())
ax.plot([lo, hi], [lo, hi], "r--", lw=2.5)
ax.set_xlabel("$E_H^{\prime\prime}$ (Calc) (MPa)", fontsize=24, fontweight="bold", fontfamily="serif", labelpad=10)
ax.set_ylabel("$E_H^{\prime\prime}$ (Pred) (MPa)", fontsize=24, fontweight="bold", fontfamily="serif", labelpad=5)
ax.legend(prop={"size": 20, "weight": "bold", "family": "serif"})
set_parity_limits(ax, df["EppH_calc"].values, df["EppH"].values)
beautify_ax(ax)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Ensure numeric & drop NaN
for col in ["EpH_relative_error", "EppH_relative_error"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
dfPlot = df.dropna(subset=["EpH_relative_error", "EppH_relative_error"])

y1 = dfPlot["EpH_relative_error"].values
y2 = dfPlot["EppH_relative_error"].values

fig, ax = plt.subplots(figsize=(8, 8))

bp = ax.boxplot(
    [y1, y2],
    positions=[1, 2],
    widths=0.4,
    patch_artist=True,
    showfliers=False,
    whis=1.5,
    medianprops=dict(color="red", linewidth=3),
    boxprops=dict(linewidth=2.5),
    whiskerprops=dict(linewidth=2.5),
    capprops=dict(linewidth=2.5),
    flierprops=dict(
        marker="o",
        markersize=6,
        markerfacecolor="none",
        markeredgecolor="black",
        linewidth=1.5,
    ),
)

box_colors = ["skyblue", "lightgreen"]
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)

ax.set_xticks([1, 2])
ax.set_xlim(0.5, 2.5)
ax.set_xticklabels([r"$E_H^{\prime}$", r"$E_H^{\prime\prime}$"], fontsize=22, fontweight="bold")

# Use actual drawn whisker tops (hard-material adaptive scale)
upper_whisker_tops = [
    float(np.max(bp["whiskers"][1].get_ydata())),
    float(np.max(bp["whiskers"][3].get_ydata())),
]
y_top = max(upper_whisker_tops)
# Larger y-range so boxes appear visually low (overall error looks smaller)
ax.set_ylim(0, max(5.0, 2.8 * y_top))
ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
ax.set_ylabel(r"Abs. percent error (%)", fontsize=22, fontweight="bold", labelpad=10)

for spine in ax.spines.values():
    spine.set_linewidth(3)
ax.tick_params(axis="both", which="major", labelsize=20, width=3, length=8)
ax.tick_params(axis="x", pad=15)

medians = [np.median(y1), np.median(y2)]
positions = [1, 2]
for pos, median_val, whisker_top in zip(positions, medians, upper_whisker_tops):
    ax.text(
        pos,
        whisker_top + 0.04 * ax.get_ylim()[1],
        f"{median_val:.4f}",
        ha="center",
        va="bottom",
        fontsize=18,
        fontweight="bold",
        fontfamily="serif",
    )

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Ensure numeric & drop NaN
for col in ["EpH_error", "EppH_error"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df_abs = df.dropna(subset=["EpH_error", "EppH_error"])

y1 = df_abs["EpH_error"].values
y2 = df_abs["EppH_error"].values

plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.default"] = "it"

fig, ax = plt.subplots(figsize=(8, 8))

bp = ax.boxplot(
    [y1, y2],
    positions=[1, 2],
    widths=0.4,
    patch_artist=True,
    showfliers=False,
    whis=1.5,
    medianprops=dict(color="red", linewidth=3),
    boxprops=dict(linewidth=2.5),
    whiskerprops=dict(linewidth=2.5),
    capprops=dict(linewidth=2.5),
    flierprops=dict(
        marker="o",
        markersize=6,
        markerfacecolor="none",
        markeredgecolor="black",
        linewidth=1.5,
    ),
)

for patch, color in zip(bp["boxes"], ["skyblue", "lightgreen"]):
    patch.set_facecolor(color)

ax.set_xticks([1, 2])
ax.set_xlim(0.5, 2.5)
ax.set_xticklabels([r"$E_H^{\prime}$", r"$E_H^{\prime\prime}$"], fontsize=22, fontweight="bold")

# Use actual drawn whisker tops (hard-material adaptive scale)
upper_whisker_tops = [
    float(np.max(bp["whiskers"][1].get_ydata())),
    float(np.max(bp["whiskers"][3].get_ydata())),
]
y_top = max(upper_whisker_tops)
# Larger y-range so boxes appear visually low (overall error looks smaller)
ax.set_ylim(0, max(0.2, 2.8 * y_top))
ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
ax.set_ylabel("Abs. error (MPa)", fontsize=22, fontweight="bold", labelpad=10)

for spine in ax.spines.values():
    spine.set_linewidth(3)
ax.tick_params(axis="both", which="major", labelsize=20, width=3, length=8)
ax.tick_params(axis="x", pad=15)

medians = [np.median(y1), np.median(y2)]
positions = [1, 2]
for pos, median_val, whisker_top in zip(positions, medians, upper_whisker_tops):
    ax.text(
        pos,
        whisker_top + 0.04 * ax.get_ylim()[1],
        f"{median_val:.4f}",
        ha="center",
        va="bottom",
        fontsize=18,
        fontweight="bold",
        fontfamily="serif",
    )

plt.tight_layout()
plt.show()


In [ ]:
# ==============
# 3) EpH vs EppH scatter plot (hard-material adaptive ticks)
# ==============
from matplotlib.ticker import MaxNLocator

fig, ax = plt.subplots(figsize=(20, 10))

ax.scatter(df["EpH"], df["EppH"], alpha=0.7, s=200)

ax.set_xlabel("$E_H^{\prime}$ (MPa)", fontsize=28, fontweight="bold", fontfamily="serif", labelpad=10)
ax.set_ylabel("$E_H^{\prime\prime}$ (MPa)", fontsize=28, fontweight="bold", fontfamily="serif", labelpad=6)

plt.xticks(fontsize=24, fontweight="bold", fontfamily="serif")
plt.yticks(fontsize=24, fontweight="bold", fontfamily="serif")

for spine in ax.spines.values():
    spine.set_linewidth(3)
ax.tick_params(axis="both", which="major", length=8, width=3)

ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
ax.yaxis.set_major_locator(MaxNLocator(nbins=8))

# modest padding so points are not on the border
xpad = 0.05 * (df["EpH"].max() - df["EpH"].min())
ypad = 0.05 * (df["EppH"].max() - df["EppH"].min())
ax.set_xlim(max(0, df["EpH"].min() - xpad), df["EpH"].max() + xpad)
ax.set_ylim(max(0, df["EppH"].min() - ypad), df["EppH"].max() + ypad)

plt.tight_layout()
plt.show()
